# Notebook 06a — mix build (read-speech corpora to a mountable dataset)

**Purpose.** Turn the two candidate add-on corpora into one Kaggle dataset that the
mixing experiments can mount and read, and produce the manifests that say exactly
which clips each experiment uses. **No model is trained here. Accelerator: None.**

This notebook is the CPU half of notebook 06. The GPU half is
`notebooks/06-data-mixing.ipynb`, which trains and scores; it mounts what this one
publishes and never decodes a WebM or an mp3.

| artifact | what it is |
|---|---|
| `shards/_meta/clips.csv` | one row per source clip that survived cleaning, with its offsets inside a shard |
| `shards/_meta/items.csv` | training items — read-speech clips glued to ~25 s, never crossing a speaker |
| `mix_spec_<E>.json` | the composition of one experiment, reproducible |
| `mix_manifest_<E>.csv` | the item ids that experiment adds to Jember |
| `shards/*.flac` | the audio itself, 16 kHz mono, phase 2 only |

## Why this is a separate notebook from the experiments

Two reasons, and the second is the expensive one.

1. **The audio is bulk.** 4,599 WebM clips plus a curated Common Voice subset become a
   multi-GB artifact. That must never cross the user's connection — it is published
   server-side from the Output tab, exactly as `jember-segments-16k` was in notebook 02.
2. **Decoding is not GPU work.** Doing the conversion inside the training notebook would
   spend the 30 h/week GPU quota on ffmpeg, once per experiment. Here it is unmetered
   CPU, paid once.

## Phases

| phase | switch | what happens |
|---|---|---|
| **1 — feasibility** | `WRITE_AUDIO = False` | every number, no audio. Output is a few MB. Download it whole; the mix design is decided on these numbers. |
| **2 — production** | `WRITE_AUDIO = True` | identical code path plus the shards. Publish via Output → New Dataset. Do not download. |
| **3 — verification** | `notebooks/06-mix-verify.ipynb` | fresh kernel, reads the *published* dataset, independent of this file. |

The measurement lives outside the switch on purpose: phase 1 already ffprobes every clip
and packs every item, so the decisions that shape the bulk artifact — the digit policy,
the speaker cap, the item length — are settled *before* the expensive pass runs.

## What notebook 04 changed, and this notebook obeys

- **Common Voice goes first, TTS second.** The measured failure of the Jember-only
  fine-tune is the *Indonesian* side (wer_id 0.1836 → 0.2058), not the Javanese side.
  Indonesian read speech is the candidate medicine; more Javanese read speech would
  deepen the distortion. The brief's original ordering stood on the opposite premise.
- **M2 (diacritising the TTS transcripts) is dead.** The official reference uses no
  diacritics, the target moved to `text_folded`, and adding diacritics nobody scores
  would only manufacture errors. TTS transcripts are already plain, so they are
  compatible with no work at all. Experiment E2 is not built here; the reason is
  printed in cell 7 so it stays on the record.
- **Nothing here is selected on the Jember dev.** This notebook selects nothing, but it
  does refuse to put a Jember dev or test recording into a mixture.


## 0 — Dependencies

`soundfile`, `librosa`, `pandas` ship with the Kaggle image. `ffmpeg`/`ffprobe` also ship
with it and are what actually reads WebM/Opus — libsndfile cannot. The decode path is
checked for real in cell 8, before phase 2 is allowed to write anything.


In [ ]:
!which ffmpeg ffprobe 2>/dev/null || echo 'ffmpeg NOT on PATH - phase 2 cannot run'
!pip list 2>/dev/null | grep -Ei '^(librosa|soundfile|pandas|numpy) ' || echo 'listing skipped'


## 1 — Knobs, output convention, path discovery

Every decision that shapes the published artifact is a named constant here, and every one
of them is written into the digest. Discovery is content-based and depth-agnostic: this
account mounts inputs owner-scoped (`/kaggle/input/datasets/<user>/<slug>/...`), so a
hardcoded path breaks the moment a dataset is renamed.

**Read the RESOLVED block before continuing.** Three datasets must be attached:
`central-javanese-corpus`, the published `jember-segments-16k` (for `manifest.csv`,
`lexicon_v1.csv`, `scorer.py` — Jember *audio* is never read by this notebook), and a
Common Voice dataset carrying both the TSVs and the audio.

**Common Voice audio is resolved by an index, not by a path.** The first upload of this
corpus held 2,377 of the 60,804 files its own metadata names — 60k loose files, silently
truncated. The re-upload carries all 60,804, but Kaggle split it across **241 chunk
folders**, each holding a partial copy of the corpus tree: the audio sits in 235 separate
`clips/` directories and the TSVs are alone in a chunk of their own.

Both shapes are the same problem — the layout depends on how an upload happened to be
chunked, which is not something to hardcode. So cell 1 walks the mount once, builds a
`filename -> path` index, and everything downstream looks clips up by name.

The metadata is resolved the same way, **file by file** — the chunking split the TSVs too,
and the chunk holding `clip_durations.tsv` does not hold `validated.tsv`, so there is no
single directory to point at. Largest copy of each name wins, which also settles it when a
truncated dataset is left attached beside a good one. Any `.tar`/`.tar.gz` on the mount is
extracted first, so an archive Kaggle left compressed works too.


In [ ]:
# ============================================================
# CELL 1 - KNOBS, OUTPUT CONVENTION, PATH DISCOVERY
# ============================================================
import os, sys, re, csv, json, math, time, random, shutil, hashlib, subprocess
import collections, itertools, tarfile
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

# ============================================================
# RUN PHASE - one switch, two passes.
#   phase 1  review : WRITE_AUDIO = False   ~20 min, a few MB   - download and read
#   phase 2  produce: WRITE_AUDIO = True    ~60 min, ~2-3 GB    - published, not downloaded
#   phase 3         : notebooks/06-mix-verify.ipynb, against the published dataset
#
# PHASE 2 as of 4 Sep 2026: phase 1 was reviewed clean (18 checks, 9,851 clips, 2,552
# items, 91 shards), so this now writes the audio. Accelerator stays None - this cell
# path is ffmpeg, not tensors. Do NOT download the output; publish it from the Output
# tab as `jv-mix-16k` with the licence field set to CC BY-SA 4.0.
# ============================================================
WRITE_AUDIO = True

# DEBUGGING KNOB, NOT A PHASE. An int keeps only the first N clips of each source and
# runs the whole path in about a minute. Phase 1 already exercises this exact code on the
# full corpora, so this buys nothing except a fast loop while chasing a crash. Its output
# goes to findings/<id>-dryrun/, so a partial build can never be unzipped over the real
# artifacts by accident.
DEBUG_N_CLIPS = None

# Paste the struct_hash printed by phase 1 here before running phase 2. Phase 2 then
# asserts it built the SAME clips and the SAME grouping that was reviewed. Audio offsets
# legitimately differ between passes (phase 1 has no decoded audio), so the fingerprint
# covers the STRUCTURE - clip ids, item assignment, order - not bytes.
EXPECT_STRUCT_HASH = "8640e1cfc11b794c35bdc06ce21390464568630fa761d3bbbc0886cb6302d888"

SEED = 20260902
random.seed(SEED); np.random.seed(SEED)

# ---- cleaning policy: Common Voice -----------------------------------------
# From notebook 01 (C4, C6) and the brief. Each of these throws data away, so each one
# prints how much.
CV_DROP_DOWNVOTED   = True   # 8.6 % of `validated` carries at least one down-vote
CV_DEDUP_SENTENCES  = True   # 30,465 clips, 12,170 unique sentences - a 2.5x repeat factor
CV_MAX_CLIPS_PER_SPEAKER = 100   # the top 10 speakers hold 59.1 % of the hours
CV_KEEP_ACCENT_JV_ALWAYS = True  # the 439 Javanese-accented clips bypass the cap

# ---- cleaning policy: text --------------------------------------------------
# The official reference NEVER writes numerals (the 28 Aug dev-set analysis), and NB 04
# measured 12.5 % of zero-shot hypotheses containing digits. Training on sentences that
# DO contain numerals teaches exactly the habit NB 08 then has to undo. "drop" removes
# those clips; "keep" exists only so phase 1 can price the loss.
DIGIT_POLICY = "drop"        # "drop" | "keep"
MIN_WORDS    = 2             # a one-word read-speech clip is not worth a 30 s window

# ---- item packing -----------------------------------------------------------
# Read-speech clips are short (CV median 3.96 s, TTS median 6.03 s) and Whisper pads every
# input to 30 s regardless, so raw clips waste most of the encoder AND pull the model's
# length habit away from the 24.9 s official dev. Glue them, exactly as NB 04's
# SEGMENT_MODE="concat25" glues Jember segments.
SR            = 16000
TARGET_S      = 25.0   # same target as NB 04's concat25
HARD_CAP_S    = 29.0   # Whisper's window is 30 s; leave headroom
JOIN_SIL_S    = 0.30   # silence inserted BETWEEN clips inside one item
ITEM_GAP_S    = 0.50   # silence between items inside one shard
SHARD_TARGET_S = 600.0 # ~10 min per FLAC. See the note below on file counts.

# NB 02 learned this the expensive way: Kaggle's dataset builder truncated a per-segment
# upload at 500 files, and loose clips are slow to mount even where they work. So the
# audio ships as a few hundred shards addressed by sample offset, never as 30k clips.
MAX_SHARD_FILES = 400

# ---- cost model, measured in NB 04 - not a guess ----------------------------
# concat25 / 1 epoch / effective batch 16 ran 273 steps in 0.68 h => ~9.0 s per optimizer
# step, consistent with the sanity run's 2.52 s per micro-batch of 4.
SEC_PER_STEP     = 9.0
EFFECTIVE_BATCH  = 16
EVAL_OVERHEAD_H  = 0.35   # official dev 372 + Jember dev subset, per run

FINDINGS = {}
# NOTE: re-running THIS cell discards everything the later cells collected. Run All.

def rec(section, **kw):
    FINDINGS.setdefault(section, {}).update(kw)

def head(title):
    print("\n" + "=" * 74); print(title); print("=" * 74)

def _num(x):
    # Significant digits, not decimal places. NB 04 lost a 6e-06 learning rate to
    # round(x, 4) and recorded it as 0.0 throughout its digest.
    x = float(x)
    if x == 0 or not np.isfinite(x): return x
    return float("%.6g" % x)

def jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return _num(float(o))
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, float):          return _num(o)
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [jsonable(v) for v in o]
    if isinstance(o, Path): return str(o)
    return o

def desc(a, name=""):
    a = np.asarray([x for x in a if x is not None and not (isinstance(x, float) and math.isnan(x))],
                   dtype=float)
    if a.size == 0: return dict(n=0)
    q = lambda p: round(float(np.percentile(a, p)), 3)
    d = dict(n=int(a.size), mean=round(float(a.mean()), 3), min_s=q(0), p25=q(25),
             median=q(50), p75=q(75), p95=q(95), max_s=q(100),
             total_h=round(float(a.sum()) / 3600, 3))
    if name:
        print("%-26s n=%-7d med=%.2f p95=%.2f max=%.2f  sum=%.2f h"
              % (name, d["n"], d["median"], d["p95"], d["max_s"], d["total_h"]))
    return d

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

# ============================================================
# OUTPUT CONVENTION - mirrors the local repo layout
# ============================================================
NOTEBOOK_ID  = "06-mix-build"
WORKING      = Path("/kaggle/working")
RUN_TAG      = "-dryrun" if DEBUG_N_CLIPS else ""
OUT_DIR      = WORKING / "findings" / (NOTEBOOK_ID + RUN_TAG)
ARTIFACT_DIR = OUT_DIR / "_kaggle-artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def out(filename):
    p = OUT_DIR / filename
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

SHARD_DIR = out("shards")
META_DIR  = out("shards/_meta")
SHARD_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

# Tee stdout to a log file. In Jupyter the live stream is sys.stdout, not sys.__stdout__ -
# writing to the latter sends the output to the kernel process instead of the cell.
_ORIG_STDOUT = getattr(sys, "_nb_orig_stdout", sys.stdout)
sys._nb_orig_stdout = _ORIG_STDOUT

class _Tee:
    def __init__(self, path, stream):
        self.f = open(path, "w", encoding="utf-8"); self.s = stream
    def write(self, t):
        self.s.write(t); self.f.write(t); return len(t)
    def flush(self):
        try: self.s.flush()
        except Exception: pass
        self.f.flush()
    def isatty(self): return False

LOG_PATH = out("run-log.txt")
try:
    if isinstance(sys.stdout, _Tee): sys.stdout.f.close()
except Exception:
    pass
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)

PHASE = ("DEBUG" if DEBUG_N_CLIPS else ("2 PRODUCE" if WRITE_AUDIO else "1 REVIEW"))
print("=" * 74)
print("RUN PHASE  : %s     WRITE_AUDIO = %s     DEBUG_N_CLIPS = %s"
      % (PHASE, WRITE_AUDIO, DEBUG_N_CLIPS))
print("=" * 74)
if DEBUG_N_CLIPS:
    print("!! PARTIAL BUILD from %d clips per source - debugging only." % DEBUG_N_CLIPS)
    print("!! Output goes to findings/%s%s/. Nothing here is publishable."
          % (NOTEBOOK_ID, RUN_TAG))
print("output dir : %s" % OUT_DIR)
print("console log: %s  (restarts every time this cell runs)" % LOG_PATH)
print("seed       : %d" % SEED)

head("LICENCE AND ATTRIBUTION")
# The shards are a derivative work of two CC-licensed corpora. TTS Central Javanese is
# CC-BY-SA-4.0, which requires attribution AND that the derivative carry the same licence;
# Common Voice is CC0 and requires nothing, but is credited anyway. Both licences are
# satisfied by shipping this notice with the data - the licence text itself may be
# referenced by URI. Written in BOTH phases so it can be reviewed before it ships.
#
# Jember contributes NO audio and NO text to this dataset: the mix specs reference the
# Jember split by policy only, so CC-BY-NC-SA does not reach this artifact. The Jember
# derivative is `jember-segments-16k` and carries its own notice.
LICENSE_TXT = '''This dataset is a derivative work and is licensed
CC-BY-SA-4.0 (https://creativecommons.org/licenses/by-sa/4.0/),
the licence of the most restrictively licensed source it contains.

SOURCES
  TTS Central Javanese - CC-BY-SA-4.0
      https://mozilladatacollective.com/datasets/cml5bn4k900aame07u0rwidcg
  Common Voice Scripted Speech 26.0, Indonesian - CC0-1.0
      https://mozilladatacollective.com/datasets/cmqinqwef00xonr07z4vbfovw

Attribution follows the Mozilla Data Collective catalog entries above.
Provided as-is, without warranties, to the extent permitted by the licence.
'''

README_TXT = '''# jv-mix-16k

Read-speech audio prepared for the Mozilla *Lost in Transcription* Indonesian-Javanese
track. Built by `notebooks/06-mix-build.ipynb`; see LICENSE.txt for sources and terms.

## What this is
16 kHz mono FLAC shards of about ten minutes each, plus `shards/_meta/` holding a
clip-level and an item-level offset table. Individual clips are NOT stored as files:
they are addressed by sample offset inside a shard, because tens of thousands of loose
files are slow to mount and do not survive Kaggle's dataset builder.

## Changes made to the source material
  * decoded to 16 kHz mono and re-encoded as FLAC (originals: WebM/Opus and mp3)
  * clips concatenated into ~25 s training items, never crossing a speaker or a folder,
    with %.2f s of silence between clips and %.2f s between items
  * transcripts normalised with the competition's pre-scoring normalizer and folded to
    the diacritic-free space the official reference uses
  * clips removed: down-voted, duplicate-sentence, over the per-speaker cap, containing
    a numeral, or shorter than %d words
No transcript wording was edited beyond that normalisation, and no audio was altered
beyond resampling, channel mixing and concatenation.

## Contents
  shards/*.flac            the audio
  shards/_meta/clips.csv   one row per source clip, with its offsets
  shards/_meta/items.csv   training items
  shards/_meta/shards.csv  shard lengths
  mix_manifest_<E>.csv     the items each experiment adds
  mix_spec_<E>.json        each experiment's composition, reproducible
''' % (JOIN_SIL_S, ITEM_GAP_S, MIN_WORDS)

out("LICENSE.txt").write_text(LICENSE_TXT, encoding="utf-8")
out("README.md").write_text(README_TXT, encoding="utf-8")
print("wrote LICENSE.txt (%d B) and README.md (%d B)"
      % (len(LICENSE_TXT), len(README_TXT)))
print("Set the Kaggle dataset's licence field to CC BY-SA 4.0 when publishing phase 2.")
rec("licence", derivative_licence="CC-BY-SA-4.0",
    sources={"tts": "CC-BY-SA-4.0", "common_voice": "CC0-1.0"},
    jember_audio_included=False)

INPUT = Path("/kaggle/input")

head("DISCOVERY")

def shallow(base, max_depth=4):
    rows, base = [], Path(base)
    for dp, dns, fns in os.walk(base):
        d = Path(dp)
        depth = len(d.relative_to(base).parts)
        if depth > max_depth:
            dns[:] = []; continue
        if depth >= 1:
            rows.append((depth, d, len(fns)))
        if len(fns) > 200:
            dns[:] = []
    return rows

layout = shallow(INPUT) if INPUT.exists() else []
for depth, d, nf in layout[:40]:
    print("  %s%s%s" % ("    " * (depth - 1), d.name, ("  [%d files]" % nf) if nf else "/"))
if len(layout) > 40:
    print("  ... +%d more directories" % (len(layout) - 40))

# 1) Common Voice. It can arrive in any of three shapes and all three resolve here, so
# the upload decision is not frozen into this notebook:
#   a) a clips/ directory - loose files, or whatever Kaggle extracted for you
#   b) a tar/tar.gz of the corpus that Kaggle left compressed
#   c) the purpose-built cv-id-clips.tar
# The 3 Sep run found the loose-file upload holding 2,377 of the 60,804 files its own
# metadata names, silently. An archive is the safer shape: one file either arrives whole
# or fails visibly.

def _scratch_dir():
    # Must be OUTSIDE /kaggle/working: everything under working IS the notebook's output,
    # and NB 04 once offered a 16 GB download by putting scratch one directory over.
    # /kaggle/temp does not exist on the image, so create the first writable candidate.
    for c in (Path("/kaggle/temp"), Path("/tmp/nb-scratch"), Path.home() / "nb-scratch"):
        try:
            c.mkdir(parents=True, exist_ok=True)
            (c / ".w").write_text("x"); (c / ".w").unlink()
            return c
        except Exception:
            continue
    raise RuntimeError("no writable scratch directory outside /kaggle/working")

CV_TARS = sorted(set(list(INPUT.glob("**/*.tar")) + list(INPUT.glob("**/*.tar.gz"))
                     + list(INPUT.glob("**/*.tgz"))))
SCRATCH = None
if CV_TARS:
    SCRATCH = _scratch_dir() / "cv-extract"
    if SCRATCH.exists() and any(SCRATCH.iterdir()):
        print("reusing already-extracted %s" % SCRATCH)
    else:
        SCRATCH.mkdir(parents=True, exist_ok=True)
        for _t in CV_TARS:
            _t0 = time.time()
            with tarfile.open(_t) as _tf:
                try:    _tf.extractall(SCRATCH, filter="data")
                except TypeError: _tf.extractall(SCRATCH)   # python < 3.12
            print("extracted %s -> %s in %.1f s" % (_t.name, SCRATCH, time.time() - _t0))

# Discovery runs over the mount AND anything just extracted, so a corpus that arrived as
# an archive is found exactly the same way as one that arrived as a directory.
_ROOTS = [INPUT] + ([SCRATCH] if SCRATCH else [])

# The audio is resolved by an INDEX from filename to path, not by one clips/ directory.
# Kaggle's uploader split this corpus across 241 chunk folders, each holding a partial
# copy of the tree: 60,804 mp3s spread over 235 clips/ directories, with the TSVs alone in
# a chunk of their own. No single directory describes that, and a layout that depends on
# how an upload happened to be chunked is not something to hardcode. Names are unique
# across the corpus, so an index is both simpler and layout-proof.
def build_cv_index(roots):
    idx, dupes = {}, 0
    for r in roots:
        for q in r.glob("**/clips/*.mp3"):
            if q.name in idx: dupes += 1
            else:            idx[q.name] = str(q)
    if not idx:      # a packaging that does not use a clips/ directory
        for r in roots:
            for q in r.glob("**/*.mp3"):
                if q.name.startswith("common_voice"):
                    idx.setdefault(q.name, str(q))
    return idx, dupes

_t0 = time.time()
CV_INDEX, CV_DUPES = build_cv_index(_ROOTS)
print("  indexed %d Common Voice mp3 in %.1f s (%d duplicate names ignored)"
      % (len(CV_INDEX), time.time() - _t0, CV_DUPES))

# Metadata is resolved FILE BY FILE, not by picking a directory. The same chunking that
# scattered the audio also split the TSVs: the chunk holding clip_durations.tsv does not
# hold validated.tsv. Largest copy wins, because a truncated copy is smaller than a
# complete one - which also settles it when a broken dataset is left attached beside a
# good one.
def find_meta(name):
    hits = sorted({q for r in _ROOTS for q in r.glob("**/" + name)})
    if not hits:
        return None
    best = max(hits, key=lambda q: q.stat().st_size)
    if len(hits) > 1:
        print("  %-22s %d copies, taking the largest (%.1f MB)"
              % (name, len(hits), best.stat().st_size / 1e6))
    return best

CV_META = {n: find_meta(n) for n in ("validated.tsv", "clip_durations.tsv")}
for n, q in CV_META.items():
    print("  %-22s %s" % (n, q or "NOT FOUND"))
CV_DIR = CV_META["clip_durations.tsv"].parent if CV_META["clip_durations.tsv"] else None
# Kept only for reporting: the single directory holding the most of the audio.
CV_CLIPS = None
if CV_INDEX:
    _by_dir = collections.Counter(str(Path(p).parent) for p in CV_INDEX.values())
    CV_CLIPS = Path(_by_dir.most_common(1)[0][0])
    if len(_by_dir) > 1:
        print("  audio is spread over %d directories; resolved by index, not by path"
              % len(_by_dir))

# 2) Central Javanese TTS - the corpus whose folders ship a per-folder mapping.tsv
TTS_MAPPINGS = sorted(INPUT.glob("**/mapping.tsv"))
TTS_DS = Path(os.path.commonpath([str(p.parent) for p in TTS_MAPPINGS])) if TTS_MAPPINGS else None

# 3) The notebook-02 artifacts, found by a manifest.csv whose header is ours.
# `jember-segments-16k` carries them TWICE: once in the canonical tree, and once inside
# the 02-data-build-lite bundle that Kaggle auto-extracted from _kaggle-artifacts/. A
# plain sorted()[0] takes the lite copy, because "0" sorts before "m" - which the 3 Sep
# run did, harmlessly, since both come from the same publish. Harmless is not the same
# as deterministic: prefer the copy sitting beside scorer.py and recordings/, and hash
# the others against it rather than assuming they agree.
_jem_cands = []
for q in sorted(INPUT.glob("**/manifest.csv")):
    try:
        hdr = q.open(encoding="utf-8", errors="replace").readline().lower()
    except Exception:
        continue
    if "seg_id" in hdr and "text_folded" in hdr and "start_sample" in hdr:
        _jem_cands.append(q)

def _jem_rank(q):
    d = q.parent
    canonical = ((d / "scorer.py").exists() + (d / "recordings").is_dir()
                 + ("lite" not in str(d).lower()))
    return (-canonical, len(str(d)))

_jem_cands.sort(key=_jem_rank)
JEM_DIR = _jem_cands[0].parent if _jem_cands else None
if len(_jem_cands) > 1:
    _jh = {q: hashlib.sha256(q.read_bytes()).hexdigest()[:16] for q in _jem_cands}
    print("  manifest.csv copies : %d" % len(_jem_cands))
    for q in _jem_cands:
        print("     %s  %s%s" % (_jh[q], q.parent, "   <- using this one" if q.parent == JEM_DIR else ""))
    if len(set(_jh.values())) > 1:
        print("  !! the copies DIFFER. One of them is stale; settle which before trusting")
        print("  !! any number this notebook produces.")
SCORER_PY = (JEM_DIR / "scorer.py") if JEM_DIR else None
LEXICON   = (JEM_DIR / "lexicon_v1.csv") if JEM_DIR else None

# ==================================================================
# MANUAL OVERRIDE - only if a line below says NOT FOUND.
# ==================================================================
# CV_DIR       = Path("/kaggle/input/.../id")
# CV_CLIPS     = CV_DIR / "clips"
# TTS_MAPPINGS = sorted(Path("/kaggle/input/.../RECORDING TTS").glob("**/mapping.tsv"))
# JEM_DIR      = Path("/kaggle/input/.../findings/02-data-build")

print("")
print("RESOLVED:")
print("  CV_DIR       =", CV_DIR or "NOT FOUND")
print("  CV_AUDIO     =", "%d mp3 indexed, from %s"
      % (len(CV_INDEX), (", ".join(t.name for t in CV_TARS) if CV_TARS
                         else "mounted directories")) if CV_INDEX else "NOT FOUND")
print("  largest dir  =", CV_CLIPS or "-")
print("  TTS mappings =", ("%d files under %s" % (len(TTS_MAPPINGS), TTS_DS)) if TTS_MAPPINGS else "NOT FOUND")
print("  JEM_DIR      =", JEM_DIR or "NOT FOUND")
print("  scorer.py    =", SCORER_PY if (SCORER_PY and SCORER_PY.exists()) else "NOT FOUND")

MISSING = [n for n, v in [("validated.tsv", CV_META["validated.tsv"]),
                          ("clip_durations.tsv", CV_META["clip_durations.tsv"]),
                          ("CV_AUDIO", CV_INDEX or None),
                          ("TTS_MAPPINGS", TTS_MAPPINGS or None), ("JEM_DIR", JEM_DIR),
                          ("scorer.py", SCORER_PY if (SCORER_PY and SCORER_PY.exists()) else None)]
           if not v]
if MISSING:
    print("")
    print("!!! UNRESOLVED and mandatory: %s" % MISSING)
    print("!!! Attach the dataset or fill the MANUAL OVERRIDE block, then re-run.")
else:
    print("")
    print("All inputs resolved - safe to run the rest.")

FFMPEG  = shutil.which("ffmpeg")
FFPROBE = shutil.which("ffprobe")
print("  ffmpeg       =", FFMPEG or "NOT FOUND")
print("  ffprobe      =", FFPROBE or "NOT FOUND")

rec("env", notebook=NOTEBOOK_ID, phase=PHASE, seed=SEED, write_audio=bool(WRITE_AUDIO),
    debug_n_clips=DEBUG_N_CLIPS, python=sys.version.split()[0], pandas=pd.__version__,
    numpy=np.__version__, cv_dir=str(CV_DIR), tts_root=str(TTS_DS),
    n_tts_mappings=len(TTS_MAPPINGS), jember_artifacts=str(JEM_DIR),
    cv_audio_source=([t.name for t in CV_TARS] if CV_TARS else "mounted_dirs"),
    cv_meta={n: str(q) for n, q in CV_META.items()},
    cv_mp3_indexed=len(CV_INDEX), cv_mp3_duplicate_names=CV_DUPES,
    cv_audio_dirs=len({str(Path(p).parent) for p in CV_INDEX.values()}),
    ffmpeg=bool(FFMPEG), ffprobe=bool(FFPROBE), unresolved=MISSING)
rec("policy", cv_drop_downvoted=CV_DROP_DOWNVOTED, cv_dedup_sentences=CV_DEDUP_SENTENCES,
    cv_max_clips_per_speaker=CV_MAX_CLIPS_PER_SPEAKER,
    cv_keep_accent_jv_always=CV_KEEP_ACCENT_JV_ALWAYS, digit_policy=DIGIT_POLICY,
    min_words=MIN_WORDS, target_s=TARGET_S, hard_cap_s=HARD_CAP_S, join_sil_s=JOIN_SIL_S,
    item_gap_s=ITEM_GAP_S, shard_target_s=SHARD_TARGET_S, sr=SR,
    sec_per_step=SEC_PER_STEP, effective_batch=EFFECTIVE_BATCH)


## 2 — The scorer, and the Jember reference vocabulary

`scorer.py` is imported from the mount, never re-implemented: the add-on text must be
normalised by exactly the same code that scores the dev set, or the mixture trains on a
different orthography from the one being measured.

`text_folded` is the training target everywhere in this project since 28 August — the
official reference carries zero `ê`/`è`/`ě` in 17,495 tokens, so folding is not a
compromise here, it is the target space.


In [ ]:
# ============================================================
# CELL 2 - SCORER + JEMBER REFERENCE VOCABULARY
# ============================================================
head("SCORER")
sys.path.insert(0, str(JEM_DIR))
import importlib, scorer
importlib.reload(scorer)
from scorer import official_normalize, fold_diacritics, lex_key, tokenize
print("scorer.py VERSION =", scorer.VERSION, " from", SCORER_PY)

# The scorer is stdlib-only and cheap to self-test; a silently different copy in the
# mount would poison every text field this notebook writes.
try:
    _st = bool(scorer.self_test(verbose=False))
except Exception as e:
    _st = False; print("scorer self-test raised:", e)
print("scorer self-test  :", "OK" if _st else "FAILED")

def norm_folded(s):
    # The training target: official normalization, then diacritic folding.
    return fold_diacritics(official_normalize(str(s)))

head("JEMBER REFERENCE (notebook 02 artifacts)")
JEM = pd.read_csv(JEM_DIR / "manifest.csv", dtype=str, keep_default_na=False)
JEM["dur_s"] = pd.to_numeric(JEM["dur_s"], errors="coerce")
JEM_TRAIN = JEM[JEM["split"] == "train"]
JEM_DEV   = JEM[JEM["split"] == "dev"]
JEM_TEST  = JEM[JEM["split"] == "test"]
print("jember segments   : %d train / %d dev / %d test"
      % (len(JEM_TRAIN), len(JEM_DEV), len(JEM_TEST)))
JEM_TRAIN_H = float(JEM_TRAIN["dur_s"].sum() / 3600.0)
print("jember train hours: %.2f" % JEM_TRAIN_H)

# Recording ids that must NEVER appear in a mixture. This notebook does not touch Jember
# audio at all; the check in cell 10 is written against these sets so that stays true.
DEV_TEST_RECS = set(JEM_DEV["audio_filename"]) | set(JEM_TEST["audio_filename"])

# Vocabulary of the Jember TRAIN split only, on the folded key. Using dev text here would
# be leakage into a coverage number that then drives a data decision.
def vocab_of(texts):
    v = collections.Counter()
    for t in texts:
        for w in tokenize(str(t), normalize=False):
            k = lex_key(w)
            if k: v[k] += 1
    return v

JEM_VOCAB = vocab_of(JEM_TRAIN["text_folded"])
print("jember train vocab: %d types, %d tokens" % (len(JEM_VOCAB), sum(JEM_VOCAB.values())))

# The three-way origin lexicon from NB 02, used to describe what each pool adds.
LEX = scorer.load_lexicon(str(LEXICON)) if (LEXICON and LEXICON.exists()) else {}
print("lexicon_v1        : %d entries" % len(LEX))

rec("jember", n_train=len(JEM_TRAIN), n_dev=len(JEM_DEV), n_test=len(JEM_TEST),
    train_hours=JEM_TRAIN_H, vocab_types=len(JEM_VOCAB),
    vocab_tokens=int(sum(JEM_VOCAB.values())), scorer_version=scorer.VERSION,
    scorer_self_test=_st, lexicon_entries=len(LEX))


## 3 — TTS Central Javanese: read the mappings, clean, measure

Notebook 01 measured the defects; this cell executes the removals the brief lists and
prints what each one cost. Two of those defects turned out to be misdiagnosed, and the
3 Sep mount audit is what settled them.

- **The "30 transcripts pointing at audio that does not exist" do not exist either.**
  Those 30 rows carry a newline *inside* the sentence: the filename row ends at its tab
  with empty text, and the sentence lands on the line below. A table reader turns that
  orphan line into a filename that cannot be found. Read line by line and rejoined, all
  30 clips come back with their full transcript — the phase-1 run of 3 Sep dropped them
  twice over, once as missing audio and once as empty text.
- **4,599 rows, 4,549 unique names — and the 50 repeats are one whole folder.** The local
  copy of the corpus has 92 folders and no duplicates at all; the Kaggle mount has 93,
  and `recordings (93)` repeats 50 clips that already exist elsewhere. Filenames are
  content hashes, so a repeated name is the same audio. The first occurrence in a
  deterministic order is kept, or the same clip would be seen twice in one epoch.
- **Folders are the only grouping unit available** — there is no speaker metadata, and 93
  folders are not necessarily 93 people (some are nested). Items never cross a folder, so
  whatever a folder is, a training item does not glue two of them together.

Durations come from `ffprobe` on the container, not from decoding: exact, and about two
orders of magnitude cheaper. The decode itself is proven separately in cell 8.


In [ ]:
# ============================================================
# CELL 3 - TTS CENTRAL JAVANESE: mappings, cleaning, durations
# ============================================================
head("TTS - MAPPINGS")

def probe_duration_s(p):
    # Container duration. Cheap (no decode) and exact enough for packing; phase 2
    # replaces it with the true decoded length.
    try:
        r = subprocess.run([FFPROBE, "-v", "error", "-show_entries", "format=duration",
                            "-of", "default=nokey=1:noprint_wrappers=1", str(p)],
                           capture_output=True, text=True, timeout=30)
        return float(r.stdout.strip())
    except Exception:
        return float("nan")

def read_mapping(m):
    # Line-based, not pandas. 30 of the 4,630 transcript rows carry a NEWLINE inside the
    # sentence: the filename row ends at the tab with empty text, and the sentence becomes
    # an orphan line below it. Any table reader turns that orphan into a filename that
    # cannot exist - which is exactly how notebook 01's "30 transcripts point at audio
    # that does not exist" arose, and how the 3 Sep phase-1 run lost those 30 clips twice
    # over: the orphan dropped as missing audio, the filename row dropped as empty text.
    # Joining the continuation back recovers both halves.
    lines = m.open(encoding="utf-8", errors="replace").read().splitlines()
    rows, joined = [], 0
    for l in lines[1:]:
        if "\t" in l:
            fn, tx = l.split("\t", 1)
            rows.append([fn.strip(), tx.strip()])
        elif l.strip() and rows:
            rows[-1][1] = (rows[-1][1] + " " + l.strip()).strip(); joined += 1
    return rows, joined

tts_rows, tts_stats = [], collections.Counter()
for m in TTS_MAPPINGS:
    folder = m.parent
    gid = str(folder.relative_to(TTS_DS)) if TTS_DS else folder.name
    hdr = m.open(encoding="utf-8", errors="replace").readline().lower()
    if "audio_filename" not in hdr or "sentence" not in hdr:
        tts_stats["mapping_bad_header"] += 1
        print("  unexpected header:", m, hdr[:80]); continue
    rows, joined = read_mapping(m)
    tts_stats["joined_continuations"] += joined
    for i, (fn, tx) in enumerate(rows):
        tts_rows.append(dict(group_id=gid, order=i, fname=fn, text_raw=tx,
                             src_path=str(folder / fn)))
print("multi-line transcripts rejoined: %d  (nb01 mistook these for missing audio)"
      % tts_stats["joined_continuations"])
TTS = pd.DataFrame(tts_rows)
print("mapping files     : %d" % len(TTS_MAPPINGS))
print("transcript rows   : %d across %d folders" % (len(TTS), TTS["group_id"].nunique()))

# --- defect 1: transcripts pointing at absent audio -------------------------
exists = TTS["src_path"].map(lambda p: Path(p).exists())
n_missing = int((~exists).sum())
print("dropped, no audio : %d   (should be ~0: notebook 01's '30 missing' were the"
      % n_missing)
print("                       multi-line transcripts rejoined above, not absent files)")
TTS = TTS[exists].copy()

# --- defect 2: the same hash-named file in two folders ----------------------
TTS = TTS.sort_values(["group_id", "order"], kind="mergesort").reset_index(drop=True)
dup = TTS.duplicated(subset=["fname"], keep="first")
n_dup = int(dup.sum())
print("dropped, dup hash : %d   (notebook 01 measured ~50)" % n_dup)
TTS = TTS[~dup].copy()

# --- durations ---------------------------------------------------------------
if DEBUG_N_CLIPS:
    TTS = TTS.head(int(DEBUG_N_CLIPS)).copy()
print("probing %d durations with ffprobe ..." % len(TTS))
_t0, durs = time.time(), []
for i, p in enumerate(TTS["src_path"]):
    durs.append(probe_duration_s(p))
    if (i + 1) % 500 == 0:
        print("  %5d/%d  (%.1f s elapsed)" % (i + 1, len(TTS), time.time() - _t0))
TTS["dur_s"] = durs
n_nan = int(TTS["dur_s"].isna().sum())
if n_nan:
    print("!! %d clips would not probe - dropped" % n_nan)
    TTS = TTS[TTS["dur_s"].notna()].copy()
print("ffprobe took      : %.1f s" % (time.time() - _t0))

# --- text --------------------------------------------------------------------
TTS["text_official"] = TTS["text_raw"].map(official_normalize)
TTS["text_folded"]   = TTS["text_raw"].map(norm_folded)
TTS["n_words"]       = TTS["text_folded"].map(lambda s: len(s.split()))
TTS["has_digit"]     = TTS["text_raw"].map(lambda s: bool(re.search(r"\d", str(s))))
TTS["source"]        = "tts"
TTS["accent_jv"]     = False
TTS["clip_id"]       = ["tts_" + f.rsplit(".", 1)[0][:16] for f in TTS["fname"]]

print("")
print("non-ASCII in TTS text (diacritics) : %d clips"
      % int(TTS["text_folded"].map(lambda s: any(ord(c) > 127 for c in s)).sum()))
print("clips containing a digit           : %d  (policy=%s)"
      % (int(TTS["has_digit"].sum()), DIGIT_POLICY))
_d_tts = desc(TTS["dur_s"], "tts, after cleaning")

rec("tts", n_mapping_files=len(TTS_MAPPINGS), n_rows_raw=len(tts_rows),
    n_joined_continuations=int(tts_stats["joined_continuations"]),
    n_dropped_missing_audio=n_missing, n_dropped_dup_hash=n_dup, n_dropped_probe_fail=n_nan,
    n_clean=len(TTS), n_groups=int(TTS["group_id"].nunique()),
    n_with_digit=int(TTS["has_digit"].sum()), duration=_d_tts)


## 4 — Common Voice Indonesian: build the pool from `validated.tsv`

`train.tsv` is unusable — notebook 01 found it holds four speakers. The pool is built
from `validated.tsv` with the brief's filters, each applied in a fixed order so the result
is reproducible, and each printing its own cost:

1. drop `down_votes >= 1` (8.6 % of validated, and they are already-validated clips)
2. deduplicate by `sentence_id` (30,465 clips, 12,170 unique sentences — a 2.5× repeat)
3. cap clips per `client_id` (the top ten speakers hold 59.1 % of the hours)
4. digit and minimum-length policy

The 439 Javanese-accented clips are flagged before any filter and, by policy, bypass the
speaker cap — they are the smallest and most on-domain subset in the whole corpus and
experiment E3 is built entirely from them.


In [ ]:
# ============================================================
# CELL 4 - COMMON VOICE: pool construction from validated.tsv
# ============================================================
head("COMMON VOICE - POOL")
CV = pd.read_csv(CV_META["validated.tsv"], sep="\t", dtype=str, keep_default_na=False,
                 quoting=csv.QUOTE_NONE)
print("validated.tsv rows: %d   speakers: %d" % (len(CV), CV["client_id"].nunique()))

DUR = pd.read_csv(CV_META["clip_durations.tsv"], sep="\t", dtype=str, keep_default_na=False)
DUR.columns = [c.strip() for c in DUR.columns]
_dcol = [c for c in DUR.columns if "duration" in c.lower()][0]
DURMAP = dict(zip(DUR[DUR.columns[0]], pd.to_numeric(DUR[_dcol], errors="coerce")))
CV["dur_s"] = CV["path"].map(lambda p: DURMAP.get(p, float("nan")) / 1000.0)
print("clips with a duration: %d of %d" % (int(CV["dur_s"].notna().sum()), len(CV)))

# ---- what the mount ACTUALLY holds -----------------------------------------
# The 2 Sep phase-1 run found every probed Common Voice clip absent from the mount while
# the TSV listed it, because only the TTS side was ever checked for missing audio. A pool
# built from a TSV that names files the dataset does not contain fails in phase 2, hours
# in, after the cheap pass said everything was fine.
CV_AVAIL = set(CV_INDEX)          # built in cell 1, layout-proof
print("mp3 present on mount : %d" % len(CV_AVAIL))
print("mp3 named by metadata: %d" % len(DURMAP))
if len(CV_AVAIL) < 0.98 * len(DURMAP):
    print("!! the mount holds %.1f%% of the clips the corpus metadata names."
          % (100.0 * len(CV_AVAIL) / max(len(DURMAP), 1)))
    print("!! This notebook can only use what is mounted; the pool shrinks accordingly.")
    print("!! If that is not intended, the Kaggle dataset needs re-uploading.")

# Javanese-accented clips, flagged BEFORE any filter. Notebook 01 counted 439 from three
# exact labels (`Javanese` 154, `jawa` 53, and the combined Jawa Tengah|medhok|surakarta
# 232). The substring rule below also catches the compound labels those three miss -
# `Javanese|Arabic`, `jawa|Ngapak`, `Javanese|medhok|East java`, `Jawa Tengah`,
# `Central Java` - which are Javanese-accented by any reading. It deliberately does NOT
# match `Betawi` (509) or `Jakartan Standard` (67). The breakdown is printed so the
# number stays auditable rather than being a bare count.
_ACC_JV = ("javanese", "jawa", "central java", "medhok", "surakarta", "ngapak")
CV["accent_jv"] = CV["accents"].map(lambda s: any(k in str(s).lower() for k in _ACC_JV))
print("accent-Javanese      : %d clips  (notebook 01's narrower rule gave 439)"
      % int(CV["accent_jv"].sum()))
for lab, n in collections.Counter(CV.loc[CV["accent_jv"], "accents"]).most_common(12):
    print("     %5d  %s" % (n, str(lab)[:70]))

n0 = len(CV)
step = []
def drop_to(mask, label):
    global CV
    before = len(CV); CV = CV[mask].copy(); after = len(CV)
    print("%-34s -%6d  -> %6d left" % (label, before - after, after))
    step.append(dict(filter=label, dropped=before - after, left=after))

drop_to(CV["dur_s"].notna(), "no duration in clip_durations")
# Availability first, so the speaker cap further down selects from clips that exist
# rather than spending its budget on names that will fail to decode.
drop_to(CV["path"].isin(CV_AVAIL), "audio not present on the mount")
if CV_DROP_DOWNVOTED:
    # The down-vote filter is about annotation quality, so nothing bypasses it - an
    # accent-Javanese clip with a disputed transcript is still a disputed transcript.
    dv = pd.to_numeric(CV["down_votes"], errors="coerce").fillna(0)
    drop_to(dv < 1, "down_votes >= 1")
if CV_DEDUP_SENTENCES:
    CV = CV.sort_values(["sentence_id", "path"], kind="mergesort")
    keep = ~CV.duplicated(subset=["sentence_id"], keep="first")
    # An accent-jv clip is never dropped as a duplicate; it is too scarce to spend.
    keep = keep | (CV["accent_jv"] if CV_KEEP_ACCENT_JV_ALWAYS else False)
    drop_to(keep, "duplicate sentence_id")

# Speaker cap, deterministic: first N clips of each speaker by path.
CV = CV.sort_values(["client_id", "path"], kind="mergesort")
rank = CV.groupby("client_id").cumcount()
keep = rank < int(CV_MAX_CLIPS_PER_SPEAKER)
if CV_KEEP_ACCENT_JV_ALWAYS:
    keep = keep | CV["accent_jv"]
drop_to(keep, "clips per speaker > %d" % CV_MAX_CLIPS_PER_SPEAKER)

CV["text_raw"]       = CV["sentence"]
CV["text_official"]  = CV["text_raw"].map(official_normalize)
CV["text_folded"]    = CV["text_raw"].map(norm_folded)
CV["n_words"]        = CV["text_folded"].map(lambda s: len(s.split()))
CV["has_digit"]      = CV["text_raw"].map(lambda s: bool(re.search(r"\d", str(s))))
print("clips containing a digit           : %d  (policy=%s)"
      % (int(CV["has_digit"].sum()), DIGIT_POLICY))

CV["source"]   = "cv"
CV["group_id"] = CV["client_id"].str.slice(0, 16)
CV["order"]    = range(len(CV))
CV["fname"]    = CV["path"]
CV["src_path"] = CV["path"].map(lambda p: CV_INDEX.get(p, ""))
CV["clip_id"]  = ["cv_" + p.rsplit(".", 1)[0].replace("common_voice_id_", "") for p in CV["path"]]
if DEBUG_N_CLIPS:
    CV = pd.concat([CV[CV["accent_jv"]], CV[~CV["accent_jv"]].head(int(DEBUG_N_CLIPS))])

_d_cv = desc(CV["dur_s"], "cv, after filters")
print("speakers left     : %d   accent-jv left: %d"
      % (CV["client_id"].nunique(), int(CV["accent_jv"].sum())))

rec("cv", n_validated=n0, filters=step, n_pool=len(CV),
    n_mp3_on_mount=len(CV_AVAIL), n_mp3_in_metadata=len(DURMAP),
    accent_labels=dict(collections.Counter(CV.loc[CV["accent_jv"], "accents"]).most_common(12)),
    n_speakers=int(CV["client_id"].nunique()), n_accent_jv=int(CV["accent_jv"].sum()),
    n_with_digit=int(CV["has_digit"].sum()), duration=_d_cv)


## 5 — One pool table, the text policy, and vocabulary coverage

The digit rule is the one text decision this notebook makes on its own, so it is priced
here rather than assumed. The official reference never writes numerals and notebook 04
still found digits in 12.5 % of hypotheses; training on sentences that spell numbers as
digits would deepen exactly the habit notebook 08 then has to undo with a rewriting net.

Coverage is recomputed here rather than quoted from notebook 01, because it is coverage of
*this* pool after *these* filters, which is a different number from the raw corpus.


In [ ]:
# ============================================================
# CELL 5 - POOL ASSEMBLY, TEXT POLICY, VOCABULARY COVERAGE
# ============================================================
head("POOL")
_cols = ["clip_id", "source", "group_id", "order", "fname", "src_path", "dur_s",
         "text_raw", "text_official", "text_folded", "n_words", "has_digit", "accent_jv"]
POOL = pd.concat([TTS[_cols], CV[_cols]], ignore_index=True)

before = len(POOL)
POOL = POOL[POOL["text_folded"].str.strip() != ""]
print("dropped, empty text after normalization: %d" % (before - len(POOL)))

before = len(POOL)
POOL = POOL[POOL["n_words"] >= int(MIN_WORDS)]
print("dropped, fewer than %d words           : %d" % (MIN_WORDS, before - len(POOL)))

n_digit_clips = int(POOL["has_digit"].sum())
h_digit = float(POOL.loc[POOL["has_digit"], "dur_s"].sum() / 3600.0)
if DIGIT_POLICY == "drop":
    POOL = POOL[~POOL["has_digit"]].copy()
    print("dropped, contains a digit              : %d  (%.2f h)" % (n_digit_clips, h_digit))
else:
    print("KEPT %d digit-bearing clips (%.2f h) - DIGIT_POLICY='keep'" % (n_digit_clips, h_digit))

POOL = POOL.sort_values(["source", "group_id", "order"], kind="mergesort").reset_index(drop=True)
print("")
for s, g in POOL.groupby("source"):
    desc(g["dur_s"], "pool: " + s)
desc(POOL["dur_s"], "pool: all")

head("VOCABULARY COVERAGE OF JEMBER TRAIN")
# Type coverage: what share of Jember's word types is attested at all in the pool.
# Token coverage: what share of Jember's word OCCURRENCES those types account for -
# the number that actually predicts how often the model has seen a word before.
jem_types  = set(JEM_VOCAB)
jem_tokens = sum(JEM_VOCAB.values())

def coverage(df, label):
    v = vocab_of(df["text_folded"])
    hit_types = jem_types & set(v)
    tok_cov = sum(JEM_VOCAB[t] for t in hit_types) / max(jem_tokens, 1)
    new_types = len(set(v) - jem_types)
    origins = collections.Counter(LEX.get(t, "unk") for t in v) if LEX else {}
    print("%-24s  types %6d  covers %5.1f%% of jember types, %5.1f%% of tokens   +%d new types"
          % (label, len(v), 100.0 * len(hit_types) / max(len(jem_types), 1),
             100.0 * tok_cov, new_types))
    return dict(pool_types=len(v), covered_types=len(hit_types),
                type_cov=len(hit_types) / max(len(jem_types), 1), token_cov=tok_cov,
                new_types=new_types, origin_mix=dict(origins))

cov = {}
cov["tts"]       = coverage(POOL[POOL["source"] == "tts"], "tts")
cov["cv"]        = coverage(POOL[POOL["source"] == "cv"], "cv")
cov["cv_jv_acc"] = coverage(POOL[(POOL["source"] == "cv") & POOL["accent_jv"]], "cv, accent-jv only")
cov["both"]      = coverage(POOL, "tts + cv")
print("")
print("Notebook 01 measured 56.2 % / 41.5 % / 69.5 % type coverage on the RAW corpora.")
print("These are after cleaning, so they are expected to differ - large gaps are not.")

rec("pool", n_clips=len(POOL), hours=float(POOL["dur_s"].sum() / 3600.0),
    n_digit_dropped=(n_digit_clips if DIGIT_POLICY == "drop" else 0),
    by_source={s: dict(n=int(len(g)), hours=float(g["dur_s"].sum() / 3600.0))
               for s, g in POOL.groupby("source")},
    coverage=cov)


## 6 — Packing: clips → items → shards

Three constraints decide this layout, and each one is a lesson already paid for.

1. **An item is ~25 s, never more than 29.** Whisper pads every input to a 30 s window, so
   a 4 s Common Voice clip costs a full training step to teach four seconds. Notebook 04
   measured Jember native items using 18.1 % of the window; gluing them to ~25 s made the
   epoch 3.8× cheaper *and* won Gate 2. Read speech is shorter still, so the effect here
   is larger.
2. **An item never crosses a speaker or a folder.** Gluing two voices into one training
   example teaches a discontinuity that does not exist in the test distribution.
3. **The audio ships as ~10-minute shards, not as loose clips.** Kaggle's dataset builder
   truncated notebook 02's per-segment upload at 500 files. Items are addressed by sample
   offset inside a shard, exactly like Jember segments inside a recording FLAC.

Because the published audio is a continuous shard plus an offset table, **the grouping is
a manifest question, not an audio question**: every experiment in notebook 06b reuses the
same shards, and a different mixture costs zero re-encoding.


In [ ]:
# ============================================================
# CELL 6 - PACKING: clips -> items -> shards
# ============================================================
head("PACKING")
JOIN_N = int(round(JOIN_SIL_S * SR))
GAP_N  = int(round(ITEM_GAP_S * SR))

def pack(pool):
    # Items: consecutive clips of ONE group, glued to TARGET_S, hard-capped at HARD_CAP_S.
    items, stats = [], collections.Counter()
    for (src, gid), g in pool.groupby(["source", "group_id"], sort=False):
        buf, buf_s = [], 0.0
        def flush(buf):
            if not buf: return
            items.append(dict(
                item_id="%s_%s_i%04d" % (src, re.sub(r"\W+", "", str(gid))[:12], len(items)),
                source=src, group_id=gid,
                clip_ids=[c["clip_id"] for c in buf],
                text_folded=" ".join(c["text_folded"] for c in buf).strip(),
                dur_s=sum(c["dur_s"] for c in buf) + JOIN_SIL_S * (len(buf) - 1),
                n_clip=len(buf)))
        for r in g.to_dict("records"):
            if r["dur_s"] > HARD_CAP_S:
                # A single clip already over the cap goes through alone and is truncated
                # by the feature extractor exactly as it would be in native mode.
                flush(buf); buf, buf_s = [], 0.0
                stats["oversize_alone"] += 1
                items.append(dict(item_id="%s_%s_i%04d" % (src, re.sub(r"\W+", "", str(gid))[:12], len(items)),
                                  source=src, group_id=gid, clip_ids=[r["clip_id"]],
                                  text_folded=r["text_folded"], dur_s=r["dur_s"], n_clip=1))
                continue
            add = r["dur_s"] + (JOIN_SIL_S if buf else 0.0)
            if buf and (buf_s + add > TARGET_S):
                flush(buf); buf, buf_s = [], 0.0
                add = r["dur_s"]
            buf.append(r); buf_s += add
        flush(buf)
    return pd.DataFrame(items), stats

ITEMS, PACK_STATS = pack(POOL)
print("items             : %d from %d clips" % (len(ITEMS), len(POOL)))
_d_it = desc(ITEMS["dur_s"], "items")
print("clips per item    : mean %.2f  max %d" % (ITEMS["n_clip"].mean(), ITEMS["n_clip"].max()))
print("items over cap    : %d  (must be 0 except single oversize clips: %d)"
      % (int((ITEMS["dur_s"] > HARD_CAP_S).sum()), PACK_STATS["oversize_alone"]))
print("window used       : %.1f%%  (native read-speech clips would be %.1f%%)"
      % (100.0 * ITEMS["dur_s"].mean() / 30.0, 100.0 * POOL["dur_s"].mean() / 30.0))

# ---- shards ----------------------------------------------------------------
head("SHARDS")
shard_of, shard_rows = {}, []
cur, cur_s, idx, cur_src = [], 0.0, 0, None
def close_shard():
    global cur, cur_s, idx
    if not cur: return
    name = "%s_%04d.flac" % (cur_src, idx)
    for it in cur: shard_of[it] = name
    shard_rows.append(dict(shard=name, n_item=len(cur), dur_s=cur_s))
    idx += 1; cur, cur_s = [], 0.0

for r in ITEMS.to_dict("records"):
    if cur_src is not None and r["source"] != cur_src:
        close_shard(); idx = 0
    cur_src = r["source"]
    add = r["dur_s"] + (ITEM_GAP_S if cur else 0.0)
    if cur and cur_s + add > SHARD_TARGET_S:
        close_shard(); add = r["dur_s"]
    cur.append(r["item_id"]); cur_s += add
close_shard()
SHARDS = pd.DataFrame(shard_rows)
ITEMS["shard"] = ITEMS["item_id"].map(shard_of)
ITEMS["path"]  = "shards/" + ITEMS["shard"]
print("shards            : %d files, %.1f min mean, %.2f GB estimated FLAC"
      % (len(SHARDS), SHARDS["dur_s"].mean() / 60.0,
         SHARDS["dur_s"].sum() * SR * 2 * 0.60 / 1e9))
if len(SHARDS) > MAX_SHARD_FILES:
    print("!! %d shards exceeds MAX_SHARD_FILES=%d. Raise SHARD_TARGET_S before phase 2 -"
          % (len(SHARDS), MAX_SHARD_FILES))
    print("!! notebook 02 hit a 500-file truncation in Kaggle's dataset builder.")

# ---- planned offsets --------------------------------------------------------
# Phase 1 has no decoded audio, so these are computed from the probed durations. Phase 2
# overwrites them with the true post-decode offsets. Both passes emit the same columns,
# and cell 10 compares the STRUCTURE (which clip in which item in which shard), not these.
clip_rows, item_rows = [], []
DURMAP_POOL = dict(zip(POOL["clip_id"], POOL["dur_s"]))
POOLMAP = POOL.set_index("clip_id").to_dict("index")
for shard, g in ITEMS.groupby("shard", sort=False):
    cursor = 0
    for it in g.to_dict("records"):
        it_start = cursor
        for j, cid in enumerate(it["clip_ids"]):
            if j: cursor += JOIN_N
            n = int(round(DURMAP_POOL[cid] * SR))
            c = POOLMAP[cid]
            clip_rows.append(dict(clip_id=cid, item_id=it["item_id"], source=it["source"],
                                  group_id=it["group_id"], shard=shard,
                                  path="shards/" + shard, start_sample=cursor,
                                  end_sample=cursor + n, dur_s=n / float(SR),
                                  accent_jv=bool(c["accent_jv"]), n_words=int(c["n_words"]),
                                  src_name=c["fname"], text_folded=c["text_folded"]))
            cursor += n
        item_rows.append(dict(item_id=it["item_id"], source=it["source"], group_id=it["group_id"],
                              shard=shard, path="shards/" + shard, start_sample=it_start,
                              end_sample=cursor, dur_s=(cursor - it_start) / float(SR),
                              n_clip=it["n_clip"], accent_jv=bool(all(POOLMAP[c]["accent_jv"] for c in it["clip_ids"])),
                              clip_ids=" ".join(it["clip_ids"]), text_folded=it["text_folded"]))
        cursor += GAP_N

CLIPS_OUT = pd.DataFrame(clip_rows)
ITEMS_OUT = pd.DataFrame(item_rows)
CLIPS_OUT.to_csv(META_DIR / "clips.csv", index=False, quoting=csv.QUOTE_ALL)
ITEMS_OUT.to_csv(META_DIR / "items.csv", index=False, quoting=csv.QUOTE_ALL)
SHARDS.to_csv(META_DIR / "shards.csv", index=False, quoting=csv.QUOTE_ALL)
print("wrote _meta/clips.csv (%d rows), items.csv (%d), shards.csv (%d)"
      % (len(CLIPS_OUT), len(ITEMS_OUT), len(SHARDS)))

# The structure fingerprint: clip ids, their item, their shard, in order. Everything a
# review of phase 1 actually depends on, and nothing that legitimately moves in phase 2.
STRUCT_HASH = hashlib.sha256(
    ("\n".join("%s|%s|%s" % (r.clip_id, r.item_id, r.shard)
               for r in CLIPS_OUT.itertuples())).encode("utf-8")).hexdigest()
print("struct_hash       : %s" % STRUCT_HASH)

rec("packing", n_items=len(ITEMS_OUT), n_clips=len(CLIPS_OUT), n_shards=len(SHARDS),
    item_duration=_d_it, mean_clips_per_item=float(ITEMS["n_clip"].mean()),
    oversize_alone=int(PACK_STATS["oversize_alone"]),
    window_used_pct=float(100.0 * ITEMS["dur_s"].mean() / 30.0),
    window_used_pct_native=float(100.0 * POOL["dur_s"].mean() / 30.0),
    est_flac_gb=float(SHARDS["dur_s"].sum() * SR * 2 * 0.60 / 1e9),
    struct_hash=STRUCT_HASH)


## 7 — The experiments, as specifications

An experiment is a *selection over `items.csv`* plus a policy for the Jember side. Nothing
here re-encodes audio, so adding or re-cutting an experiment later is free.

Ordering follows notebook 04, not the original brief. The measured failure of the
Jember-only fine-tune is the Indonesian side (`wer_id` 0.1836 → 0.2058) and the untouched
`unk` class (0.3944 over 2,396 tokens, a fifth of all error mass). Indonesian read speech
is the candidate remedy for both; more Javanese read speech is not.

| id | adds | why it is in this order |
|---|---|---|
| **E3** | the Javanese-accented Common Voice clips only | smallest, closest to domain, cheapest run in the set |
| **E4** | the curated Common Voice pool | the real test of the Indonesian-side hypothesis |
| **E6** | E4 with Jember upsampled ×2 | keeps in-domain data from being outnumbered |
| **E1** | TTS Central Javanese | the brief's first arm, demoted: it deepens the side that already won |
| **E5** | E4 then Jember alone (curriculum) | often beats flat mixing on small in-domain data |

**E2 is deliberately not built.** It was "diacritise the TTS transcripts with a
text-to-text model, then mix in diacritic space". The official reference carries zero
diacritics, the training target moved to `text_folded`, so E2 would spend a model to
manufacture a distinction nothing scores. Its one surviving benefit — that TTS text needs
no conversion — arrives for free.


In [ ]:
# ============================================================
# CELL 7 - EXPERIMENT SPECIFICATIONS
# ============================================================
head("EXPERIMENTS")

# From NB 04's concat25 run, measured, not derived here: the Jember train split becomes
# 1,455 items at TARGET_S=25 and trains 273 optimizer steps at effective batch 16 in
# 0.68 h. Notebook 06b recomputes this from the manifest; it is a constant here only so
# the cost table below can exist before any GPU is touched.
JEMBER_CONCAT25_ITEMS = 1455
JEMBER_BASELINE_H     = 0.68

# How many hours of Common Voice E4 should use. The pool is deliberately larger than any
# single experiment needs, so this is a selection, not a rebuild: speakers are taken in a
# fixed order until the target is met.
E4_TARGET_H = 10.0

def take_hours(df, target_h, group_col="group_id"):
    # Whole groups, deterministic order, until the target is reached. Whole groups because
    # half a speaker is a strange thing to have in a corpus and an awkward thing to report.
    keep, acc = [], 0.0
    for gid, g in df.groupby(group_col, sort=True):
        if acc >= target_h * 3600: break
        keep.append(gid); acc += float(g["dur_s"].sum())
    return df[df[group_col].isin(keep)].copy()

IT = ITEMS_OUT
E4_ITEMS = take_hours(IT[IT["source"] == "cv"], E4_TARGET_H)

SPECS = {
  "E3": dict(question="Do the Javanese-accented Indonesian clips help? (~500 clips, 15 speakers)",
             items=IT[(IT["source"] == "cv") & IT["accent_jv"]],
             jember=dict(split="train", segment_mode="concat25", upsample=1), curriculum=False),
  "E4": dict(question="Does curated Common Voice repair the Indonesian regression?",
             items=E4_ITEMS,
             jember=dict(split="train", segment_mode="concat25", upsample=1), curriculum=False),
  "E6": dict(question="Same as E4, with Jember upsampled so in-domain data is not outnumbered.",
             items=E4_ITEMS,
             jember=dict(split="train", segment_mode="concat25", upsample=2), curriculum=False),
  "E1": dict(question="Does TTS Central Javanese help at all?",
             items=IT[IT["source"] == "tts"],
             jember=dict(split="train", segment_mode="concat25", upsample=1), curriculum=False),
  "E5": dict(question="Curriculum: pre-fine-tune on the addition, then Jember alone.",
             items=E4_ITEMS,
             jember=dict(split="train", segment_mode="concat25", upsample=1), curriculum=True),
}

rows = []
for eid, sp in SPECS.items():
    it = sp["items"]
    add_h  = float(it["dur_s"].sum() / 3600.0)
    # Speaker concentration. The accent-Javanese subset is only ~15 speakers and one of
    # them holds nearly half of it, so a win for E3 could be adaptation to one voice
    # rather than to Javanese-accented Indonesian. That is not a reason to skip the arm -
    # it is a reason to know the number before reading the result.
    _by_g = it.groupby("group_id")["dur_s"].sum()
    n_groups = int(len(_by_g))
    top_share = float(_by_g.max() / max(_by_g.sum(), 1e-9))
    sp["concentration"] = dict(n_groups=n_groups, top_group_share=top_share)
    n_jem  = JEMBER_CONCAT25_ITEMS * sp["jember"]["upsample"]
    n_tot  = len(it) + n_jem
    steps  = math.ceil(n_tot / EFFECTIVE_BATCH)
    hours  = steps * SEC_PER_STEP / 3600.0 + EVAL_OVERHEAD_H
    if sp["curriculum"]:
        # Two stages: the addition alone, then Jember alone. Same total data, two runs.
        steps = math.ceil(len(it) / EFFECTIVE_BATCH) + math.ceil(JEMBER_CONCAT25_ITEMS / EFFECTIVE_BATCH)
        hours = steps * SEC_PER_STEP / 3600.0 + 2 * EVAL_OVERHEAD_H
    rows.append(dict(exp=eid, n_add_items=len(it), add_hours=add_h,
                     n_add_clips=int(it["n_clip"].sum()), n_speakers=n_groups,
                     top_speaker_share=top_share, jember_upsample=sp["jember"]["upsample"],
                     n_total_items=n_tot, steps=steps, est_gpu_h=hours,
                     add_share_of_items=len(it) / max(n_tot, 1)))
    sp["items"][["item_id", "source", "group_id", "shard", "path", "start_sample",
                 "end_sample", "dur_s", "n_clip", "text_folded"]].to_csv(
        out("mix_manifest_%s.csv" % eid), index=False, quoting=csv.QUOTE_ALL)
    json.dump(jsonable(dict(exp=eid, question=sp["question"], jember=sp["jember"],
                            curriculum=sp["curriculum"], n_add_items=len(it),
                            add_hours=add_h, concentration=sp["concentration"],
                            per_speaker_hours={str(k): float(v / 3600.0)
                                               for k, v in _by_g.sort_values(ascending=False).head(20).items()},
                            seed=SEED, struct_hash=STRUCT_HASH,
                            policy=FINDINGS["policy"])),
              open(out("mix_spec_%s.json" % eid), "w"), indent=2)

COST = pd.DataFrame(rows)
print(COST.to_string(index=False,
      formatters={"add_hours": "{:.2f}".format, "est_gpu_h": "{:.2f}".format,
                  "add_share_of_items": "{:.1%}".format,
                  "top_speaker_share": "{:.1%}".format}))
for r in rows:
    if r["top_speaker_share"] > 0.30:
        print("!! %s: one speaker holds %.0f%% of the added audio across %d speakers."
              % (r["exp"], 100 * r["top_speaker_share"], r["n_speakers"]))
        print("!! Readable either way, but if this arm wins, the follow-up question is")
        print("!! whether the win survives dropping that speaker.")
print("")
print("Jember-only baseline for comparison: %d items, %d steps, %.2f h (measured, NB 04)"
      % (JEMBER_CONCAT25_ITEMS, math.ceil(JEMBER_CONCAT25_ITEMS / EFFECTIVE_BATCH),
         JEMBER_BASELINE_H))
print("Whole set, run once each     : %.2f GPU h" % COST["est_gpu_h"].sum())
print("Weekly quota                 : 30 h. Budget is not the binding constraint here;")
print("the binding constraint is that every arm must clear the three keep-rules.")
print("")
print("E2 (M2 diacritisation) is intentionally absent - see this cell's markdown.")

rec("experiments", e4_target_h=E4_TARGET_H, jember_concat25_items=JEMBER_CONCAT25_ITEMS,
    table=COST.to_dict("records"), total_est_gpu_h=float(COST["est_gpu_h"].sum()),
    e2_dropped_reason="official reference has zero diacritics; target is text_folded")


## 8 — Decode probe

Phase 2 decodes tens of thousands of files. This proves the decode path on a sample first,
prices the full pass, and checks the one thing `ffprobe` cannot: that the container
duration and the decoded sample count agree. A systematic mismatch would put every offset
in `clips.csv` slightly wrong, and nothing downstream would notice.

Loudness is deliberately **not** normalised. Notebook 02 wrote Jember FLACs at native
level, and normalising only the add-on corpora would make loudness a proxy for source —
a cue the model can learn instead of learning the language.


In [ ]:
# ============================================================
# CELL 8 - DECODE PROBE
# ============================================================
head("DECODE PROBE")
PEAK_NORMALIZE = False   # see the markdown above before changing this

def decode_16k(path):
    # ffmpeg is the only decoder on the image that reads WebM/Opus; libsndfile does not.
    # Mono, 16 kHz, float32 - one command, no intermediate file, deterministic.
    cmd = [FFMPEG, "-v", "error", "-nostdin", "-i", str(path),
           "-f", "f32le", "-acodec", "pcm_f32le", "-ac", "1", "-ar", str(SR), "-"]
    r = subprocess.run(cmd, capture_output=True)
    if r.returncode != 0 or not r.stdout:
        raise RuntimeError((r.stderr or b"")[:300].decode("utf-8", "replace") or "empty decode")
    return np.frombuffer(r.stdout, dtype=np.float32)

N_PROBE = 24
# Sampled, not head(). The 2 Sep run probed the first 24 clips of each source, which for
# Common Voice meant a handful of consecutive speakers - so a gap in the mount read as a
# total decode failure, and a total failure would have read the same way. A seeded random
# sample across the pool distinguishes the two.
_rs = np.random.RandomState(SEED)
def _sample(df, n):
    if len(df) == 0: return df
    idx = _rs.choice(len(df), size=min(n, len(df)), replace=False)
    return df.iloc[idx]
probe = pd.concat([_sample(POOL[POOL["source"] == "tts"], N_PROBE),
                   _sample(POOL[POOL["source"] == "cv"], N_PROBE)])
res, t0 = [], time.time()
for r in probe.to_dict("records"):
    try:
        a = decode_16k(r["src_path"])
        res.append(dict(source=r["source"], ok=True, probed_s=r["dur_s"],
                        decoded_s=len(a) / float(SR), peak=float(np.abs(a).max()),
                        rms=float(np.sqrt(np.mean(np.square(a.astype(np.float64)))))))
    except Exception as e:
        res.append(dict(source=r["source"], ok=False, probed_s=r["dur_s"], err=str(e)[:120]))
PR = pd.DataFrame(res)
elapsed = time.time() - t0
n_ok = int(PR["ok"].sum())
print("decoded %d/%d files in %.1f s" % (n_ok, len(PR), elapsed))
for s, g in PR.groupby("source"):
    print("  %-4s %d/%d decoded" % (s, int(g["ok"].sum()), len(g)))
for e in PR[~PR["ok"]].to_dict("records")[:5]:
    print("  FAIL:", e.get("err"))

if n_ok:
    OK = PR[PR["ok"]]
    OK = OK.assign(drift_s=(OK["decoded_s"] - OK["probed_s"]).abs())
    print("duration drift    : median %.4f s   max %.4f s"
          % (OK["drift_s"].median(), OK["drift_s"].max()))
    if OK["drift_s"].max() > 0.10:
        print("!! a decoded clip differs from its probed duration by more than 100 ms.")
        print("!! Offsets in clips.csv come from the DECODED length in phase 2, so this is")
        print("!! not a correctness bug - but a large drift means phase 1's item durations")
        print("!! and the cost table above are optimistic. Re-read before phase 2.")
    for s, g in OK.groupby("source"):
        print("  %-4s peak median %.3f   rms median %.4f" % (s, g["peak"].median(), g["rms"].median()))
    audio_s = float(OK["decoded_s"].sum())
    rate = audio_s / max(elapsed, 1e-9)
    total_s = float(POOL["dur_s"].sum())
    print("")
    print("throughput        : %.0fx realtime" % rate)
    print("phase 2 projection: %.1f min of decoding for %.2f h of audio"
          % (total_s / rate / 60.0, total_s / 3600.0))
    print("phase 2 output    : ~%.2f GB of FLAC in %d files"
          % (SHARDS["dur_s"].sum() * SR * 2 * 0.60 / 1e9, len(SHARDS)))
else:
    rate = float("nan")
    print("!! nothing decoded - phase 2 cannot run. Check that ffmpeg is on PATH.")

rec("decode_probe", n=len(PR), n_ok=n_ok, elapsed_s=elapsed,
    realtime_factor=(None if not n_ok else float(rate)),
    drift_median_s=(None if not n_ok else float(OK["drift_s"].median())),
    drift_max_s=(None if not n_ok else float(OK["drift_s"].max())),
    proj_decode_min=(None if not n_ok else float(POOL["dur_s"].sum() / rate / 60.0)),
    peak_normalize=PEAK_NORMALIZE,
    fails=[e.get("err") for e in PR[~PR["ok"]].to_dict("records")][:10])


## 9 — Write the shards (phase 2 only)

Identical code path to phase 1 up to this point; this is the only cell the switch gates.
Each shard is decoded clip by clip, glued with the silences declared in cell 1, and written
once as a 16 kHz mono FLAC. `clips.csv` and `items.csv` are then rewritten with the **true**
post-decode offsets, replacing the planned ones.

A clip that probed fine but fails to decode is dropped and recorded in
`decode_failures.csv` rather than replaced with silence — training on silence labelled with
a sentence is worse than having one clip fewer. Any such drop changes the structure, so it
is reported against `EXPECT_STRUCT_HASH` explicitly instead of being absorbed quietly.


In [ ]:
# ============================================================
# CELL 9 - WRITE SHARDS  (phase 2 only)
# ============================================================
head("AUDIO")
FAILED, WROTE = [], []
if not WRITE_AUDIO:
    print("WRITE_AUDIO = False - phase 1. No audio written.")
    print("Everything above was computed on the full corpora anyway; that is the point of")
    print("phase 1. Review the numbers, then set WRITE_AUDIO = True and re-run.")
else:
    import soundfile as sf
    t0 = time.time()
    new_clip_rows, new_item_rows, shard_len = [], [], {}
    ITEM_BY_SHARD = {s: g.to_dict("records") for s, g in ITEMS.groupby("shard", sort=False)}
    for k, (shard, its) in enumerate(ITEM_BY_SHARD.items()):
        buf, cursor = [], 0
        for it in its:
            it_start = cursor
            got = 0
            for j, cid in enumerate(it["clip_ids"]):
                c = POOLMAP[cid]
                try:
                    a = decode_16k(c["src_path"])
                except Exception as e:
                    FAILED.append(dict(clip_id=cid, item_id=it["item_id"], shard=shard,
                                       src_path=c["src_path"], err=str(e)[:200]))
                    continue
                a = np.clip(np.nan_to_num(a), -1.0, 1.0)
                if PEAK_NORMALIZE and np.abs(a).max() > 0:
                    a = a * (0.95 / np.abs(a).max())
                if got: buf.append(np.zeros(JOIN_N, dtype=np.float32)); cursor += JOIN_N
                buf.append(a); n = len(a)
                new_clip_rows.append(dict(clip_id=cid, item_id=it["item_id"], source=it["source"],
                                          group_id=it["group_id"], shard=shard,
                                          path="shards/" + shard, start_sample=cursor,
                                          end_sample=cursor + n, dur_s=n / float(SR),
                                          accent_jv=bool(c["accent_jv"]), n_words=int(c["n_words"]),
                                          src_name=c["fname"], text_folded=c["text_folded"]))
                cursor += n; got += 1
            if got == 0:
                continue   # every clip of this item failed; the item ceases to exist
            kept = [r["clip_id"] for r in new_clip_rows[-got:]]
            new_item_rows.append(dict(item_id=it["item_id"], source=it["source"],
                                      group_id=it["group_id"], shard=shard,
                                      path="shards/" + shard, start_sample=it_start,
                                      end_sample=cursor, dur_s=(cursor - it_start) / float(SR),
                                      n_clip=got,
                                      accent_jv=bool(all(POOLMAP[c]["accent_jv"] for c in kept)),
                                      clip_ids=" ".join(kept),
                                      text_folded=" ".join(POOLMAP[c]["text_folded"] for c in kept)))
            buf.append(np.zeros(GAP_N, dtype=np.float32)); cursor += GAP_N
        if not buf:
            continue
        y = np.concatenate(buf).astype(np.float32)
        sf.write(str(SHARD_DIR / shard), y, SR, format="FLAC", subtype="PCM_16")
        shard_len[shard] = len(y)
        WROTE.append(shard)
        if (k + 1) % 20 == 0 or k + 1 == len(ITEM_BY_SHARD):
            print("  %4d/%d shards   %.1f min elapsed"
                  % (k + 1, len(ITEM_BY_SHARD), (time.time() - t0) / 60.0))

    CLIPS_OUT = pd.DataFrame(new_clip_rows)
    ITEMS_OUT = pd.DataFrame(new_item_rows)
    SHARDS = pd.DataFrame([dict(shard=s, n_item=int((ITEMS_OUT["shard"] == s).sum()),
                                dur_s=shard_len[s] / float(SR), n_sample=shard_len[s])
                           for s in WROTE])
    CLIPS_OUT.to_csv(META_DIR / "clips.csv", index=False, quoting=csv.QUOTE_ALL)
    ITEMS_OUT.to_csv(META_DIR / "items.csv", index=False, quoting=csv.QUOTE_ALL)
    SHARDS.to_csv(META_DIR / "shards.csv", index=False, quoting=csv.QUOTE_ALL)
    if FAILED:
        pd.DataFrame(FAILED).to_csv(out("decode_failures.csv"), index=False, quoting=csv.QUOTE_ALL)
    # The experiment manifests must be rebuilt on the TRUE offsets, or notebook 06b would
    # read the planned ones and slice the wrong samples.
    for eid, sp in SPECS.items():
        sel = ITEMS_OUT[ITEMS_OUT["item_id"].isin(set(sp["items"]["item_id"]))]
        sel[["item_id", "source", "group_id", "shard", "path", "start_sample", "end_sample",
             "dur_s", "n_clip", "text_folded"]].to_csv(out("mix_manifest_%s.csv" % eid),
                                                       index=False, quoting=csv.QUOTE_ALL)
        # The SPEC has to be rewritten too. Cell 7 computed its hours from ffprobe
        # container durations; these are the decoded ones, and Opus decodes ~60 ms longer
        # than it reports, so the TTS arm drifts about 1 %. Leaving the spec stale makes
        # two files that describe the same experiment disagree - which the phase-3
        # verifier duly caught on the 5 Sep build.
        _p = out("mix_spec_%s.json" % eid)
        _spec = json.load(open(_p))
        _byg = sel.groupby("group_id")["dur_s"].sum()
        _spec.update(add_hours=float(sel["dur_s"].sum() / 3600.0), n_add_items=len(sel),
                     struct_hash=STRUCT_HASH_2, hours_source="decoded",
                     concentration=dict(n_groups=int(len(_byg)),
                                        top_group_share=float(_byg.max() / max(_byg.sum(), 1e-9))))
        json.dump(jsonable(_spec), open(_p, "w"), indent=2)
    STRUCT_HASH_2 = hashlib.sha256(
        ("\n".join("%s|%s|%s" % (r.clip_id, r.item_id, r.shard)
                   for r in CLIPS_OUT.itertuples())).encode("utf-8")).hexdigest()
    gb = sum((SHARD_DIR / s).stat().st_size for s in WROTE) / 1e9
    print("")
    print("shards written    : %d   %.2f GB   %.1f min" % (len(WROTE), gb, (time.time() - t0) / 60.0))
    print("clips written     : %d   items: %d   decode failures: %d"
          % (len(CLIPS_OUT), len(ITEMS_OUT), len(FAILED)))
    print("struct_hash       : %s" % STRUCT_HASH_2)
    if FAILED:
        print("!! %d clips failed to decode, so the structure differs from phase 1 BY DESIGN."
              % len(FAILED))
        print("!! See decode_failures.csv. phase 1 hash was %s" % STRUCT_HASH)
    STRUCT_HASH = STRUCT_HASH_2
    rec("audio", n_shards=len(WROTE), gb=gb, minutes=(time.time() - t0) / 60.0,
        n_clips=len(CLIPS_OUT), n_items=len(ITEMS_OUT), n_decode_failures=len(FAILED),
        struct_hash=STRUCT_HASH_2)


## 10 — Self-validation

Every check is a way this notebook could have failed silently. Checks that need the audio
are skipped in phase 1 and reported as skipped — phase 1 ends with `ANALYSIS PASS OK`, never
with a green build, because the gate stays shut until the artifact exists.

The size guard scans **all of `/kaggle/working`**, not just `findings/`. Notebook 04's
original guard scanned one directory and passed green while 16 GB sat next to it.


In [ ]:
# ============================================================
# CELL 10 - SELF-VALIDATION
# ============================================================
head("SELF-VALIDATION")
CHECKS = []
def check(name, ok, detail="", skipped=False):
    CHECKS.append(dict(name=name, ok=bool(ok), skipped=bool(skipped), detail=str(detail)[:200]))
    print("%-4s %-44s %s" % ("SKIP" if skipped else ("PASS" if ok else "FAIL"), name, detail))

check("inputs resolved", not MISSING, "missing=%s" % MISSING)
check("scorer self-test", _st)
check("no empty training text", int((ITEMS_OUT["text_folded"].str.strip() == "").sum()) == 0)
check("every item within the 30 s window",
      float(ITEMS_OUT["dur_s"].max()) <= 30.0, "max %.2f s" % ITEMS_OUT["dur_s"].max())
check("items over the soft cap are single clips",
      bool(((ITEMS_OUT["dur_s"] > HARD_CAP_S) & (ITEMS_OUT["n_clip"] > 1)).sum() == 0))
check("no clip used twice", int(CLIPS_OUT["clip_id"].duplicated().sum()) == 0)
check("no item spans two groups",
      int(CLIPS_OUT.groupby("item_id")["group_id"].nunique().max()) == 1)
check("no item spans two shards",
      int(CLIPS_OUT.groupby("item_id")["shard"].nunique().max()) == 1)
check("offsets are ordered and inside the shard",
      bool((CLIPS_OUT["end_sample"] > CLIPS_OUT["start_sample"]).all()))
_ov = 0
for s, g in CLIPS_OUT.sort_values(["shard", "start_sample"]).groupby("shard"):
    _ov += int((g["start_sample"].values[1:] < g["end_sample"].values[:-1]).sum())
check("no two clips overlap in a shard", _ov == 0, "%d overlaps" % _ov)
check("shard file count under the Kaggle limit", len(SHARDS) <= MAX_SHARD_FILES,
      "%d shards" % len(SHARDS))
if DIGIT_POLICY == "drop":
    check("no digits in any training text",
          int(ITEMS_OUT["text_folded"].str.contains(r"\d").sum()) == 0)
else:
    check("no digits in any training text", True, "DIGIT_POLICY='keep'", skipped=True)
# "The dev must not change" (brief, Jebakan). This notebook cannot alter the Jember dev
# directly, so the check is that nothing it emits points anywhere near Jember: every path
# is a shard of this build, and no source clip was read from the notebook-02 artifacts.
check("nothing points outside this build's shards",
      bool(CLIPS_OUT["path"].str.startswith("shards/").all()
           and ITEMS_OUT["path"].str.startswith("shards/").all()))
check("no source clip came from the Jember mount",
      not POOL["src_path"].str.startswith(str(JEM_DIR)).any(),
      "%d jember dev/test recordings exist and none are referenced" % len(DEV_TEST_RECS))
check("common voice speaker cap respected",
      int(CLIPS_OUT[CLIPS_OUT["source"] == "cv"].groupby("group_id")["clip_id"].count().max())
      <= int(CV_MAX_CLIPS_PER_SPEAKER) + int(CV["accent_jv"].sum()),
      "max %d clips for one speaker"
      % int(CLIPS_OUT[CLIPS_OUT["source"] == "cv"].groupby("group_id")["clip_id"].count().max()))
# A derivative of CC-BY-SA material that ships without its notice is a licence breach,
# and the notice is written in cell 11 - which runs after this one. So this checks the
# PREVIOUS run's files, and is expected to fail exactly once, on a first run.
check("licence notice ships with the data",
      (OUT_DIR / "LICENSE.txt").exists() and (OUT_DIR / "README.md").exists(),
      "written by cell 11; fails only before cell 11 has ever run here")
check("every experiment manifest is non-empty and reproducible",
      all((out("mix_manifest_%s.csv" % e)).exists() and len(sp["items"]) > 0
          for e, sp in SPECS.items()))
check("decode path works", int(PR["ok"].sum()) == len(PR),
      "%d/%d probe files decoded" % (int(PR["ok"].sum()), len(PR)))

if WRITE_AUDIO:
    import soundfile as sf
    bad_hdr, bad_len = [], []
    for r in SHARDS.to_dict("records"):
        p = SHARD_DIR / r["shard"]
        try:
            info = sf.info(str(p))
        except Exception as e:
            bad_hdr.append((r["shard"], str(e)[:80])); continue
        if info.samplerate != SR or info.channels != 1:
            bad_hdr.append((r["shard"], "%d Hz, %d ch" % (info.samplerate, info.channels)))
        if info.frames < int(r["n_sample"]):
            bad_len.append((r["shard"], info.frames, int(r["n_sample"])))
    check("every shard header is 16 kHz mono", not bad_hdr, str(bad_hdr[:3]))
    check("every shard is long enough for its offsets", not bad_len, str(bad_len[:3]))
    # A correct header can sit in front of digital silence, so decode a sample for real.
    rs = np.random.RandomState(SEED)
    sample = CLIPS_OUT.iloc[rs.choice(len(CLIPS_OUT), size=min(40, len(CLIPS_OUT)), replace=False)]
    silent = 0
    for r in sample.to_dict("records"):
        y, _ = sf.read(str(OUT_DIR / r["path"]), start=int(r["start_sample"]),
                       stop=int(r["end_sample"]), dtype="float32")
        if y.size == 0 or float(np.abs(y).max()) < 1e-4: silent += 1
    check("sampled clips contain actual audio", silent == 0, "%d/%d silent" % (silent, len(sample)))
    check("no decode failures", len(FAILED) == 0, "%d failed" % len(FAILED))
    if EXPECT_STRUCT_HASH:
        check("structure matches the reviewed phase 1", EXPECT_STRUCT_HASH == STRUCT_HASH,
              "expected %s..., got %s..." % (str(EXPECT_STRUCT_HASH)[:12], STRUCT_HASH[:12]))
    else:
        check("structure matches the reviewed phase 1", False,
              "EXPECT_STRUCT_HASH not set - paste phase 1's hash before phase 2", skipped=True)
else:
    for n in ("every shard header is 16 kHz mono", "every shard is long enough for its offsets",
              "sampled clips contain actual audio", "no decode failures",
              "structure matches the reviewed phase 1"):
        check(n, False, "phase 1 - no audio on disk", skipped=True)

# Size guard over ALL of /kaggle/working, not just findings/.
tot = sum(p.stat().st_size for p in WORKING.rglob("*") if p.is_file())
big = sorted(((sum(q.stat().st_size for q in d.rglob("*") if q.is_file()), d)
              for d in WORKING.iterdir() if d.is_dir()), reverse=True)[:5]
print("")
print("output tree       : %.2f GB total" % (tot / 1e9))
for sz, d in big:
    print("   %8.2f GB  %s" % (sz / 1e9, d))
limit_gb = 8.0 if WRITE_AUDIO else 0.25
check("download size is what this phase intends", tot / 1e9 <= limit_gb,
      "%.2f GB against a %.1f GB budget" % (tot / 1e9, limit_gb))

n_pass = sum(1 for c in CHECKS if c["ok"] and not c["skipped"])
n_fail = sum(1 for c in CHECKS if not c["ok"] and not c["skipped"])
n_skip = sum(1 for c in CHECKS if c["skipped"])
print("")
print("%d passed, %d failed, %d skipped" % (n_pass, n_fail, n_skip))
if n_fail == 0 and WRITE_AUDIO:
    VERDICT = "BUILD OK - publish it, then run notebooks/06-mix-verify.ipynb"
elif n_fail == 0:
    VERDICT = "ANALYSIS PASS OK - phase 1 only, nothing is built yet"
else:
    VERDICT = "FAILED - do not publish, do not train"
print(VERDICT)
rec("validation", checks=CHECKS, n_pass=n_pass, n_fail=n_fail, n_skip=n_skip,
    verdict=VERDICT, working_gb=float(tot / 1e9))


## 11 — Digest and packaging

Run last. Copy the whole block between the markers back into the chat — that is the handoff.

Phase 1's job ends there: the mix design is decided on these numbers, not on the plan.


In [ ]:
# ============================================================
# CELL 11 - DIGEST + PACKAGING
# Copy the ENTIRE block between the BEGIN/END markers back into the chat.
# ============================================================
head("FINGERPRINTS")
small = (["mix_manifest_%s.csv" % e for e in SPECS]
         + ["mix_spec_%s.json" % e for e in SPECS] + ["LICENSE.txt", "README.md"])
fps = {}
for name in small + ["shards/_meta/clips.csv", "shards/_meta/items.csv", "shards/_meta/shards.csv"]:
    p = OUT_DIR / name
    if p.exists():
        fps[name] = dict(sha256=sha256_file(p)[:16], bytes=p.stat().st_size)
        print("  %-34s %8d B  %s" % (name, p.stat().st_size, fps[name]["sha256"]))
rec("fingerprints", files=fps, struct_hash=STRUCT_HASH)

# A lite zip of just the small files, for the phase whose output is big.
lite = ARTIFACT_DIR / ("%s-lite.zip" % NOTEBOOK_ID)
import zipfile
with zipfile.ZipFile(lite, "w", zipfile.ZIP_DEFLATED) as z:
    for name in small + ["shards/_meta/clips.csv", "shards/_meta/items.csv",
                         "shards/_meta/shards.csv", "run-log.txt", "dataset_findings.json"]:
        p = OUT_DIR / name
        if p.exists(): z.write(p, name)

FIN = jsonable(FINDINGS)
json.dump(FIN, open(out("dataset_findings.json"), "w"), indent=1)
with zipfile.ZipFile(lite, "a", zipfile.ZIP_DEFLATED) as z:
    z.write(out("dataset_findings.json"), "dataset_findings.json")
print("lite zip          : %s (%.0f KB)" % (lite, lite.stat().st_size / 1024))

head("NEXT STEPS")
if not WRITE_AUDIO:
    print("Phase 1 is done. Nothing has been built.")
    print("  1. Download the whole Output (a few MB) and paste the digest into the chat.")
    print("  2. Decide the mix design on these numbers - especially E4_TARGET_H, the")
    print("     speaker cap, and DIGIT_POLICY, all three of which change what gets written.")
    print("  3. Set EXPECT_STRUCT_HASH = \"%s\"" % STRUCT_HASH)
    print("  4. Set WRITE_AUDIO = True and Run All. Do NOT download that output.")
    print("  5. Output tab -> New Dataset. Suggested slug: jv-mix-16k")
else:
    print("Phase 2 is done. Do NOT download this output.")
    print("  1. Output tab -> New Dataset -> slug jv-mix-16k. Wait for it to finish.")
    print("  2. Run notebooks/06-mix-verify.ipynb in a FRESH session against that dataset.")
    print("  3. Only when it prints DATASET VERIFIED does notebook 06b get to start.")
print("")
print("=" * 74)
print("BEGIN CLAUDE-HANDOFF-JSON")
print(json.dumps(FIN, ensure_ascii=False, separators=(",", ":")))
print("END CLAUDE-HANDOFF-JSON")
print("=" * 74)
sys.stdout.flush()
